In [1]:
DATASET_SIZE = 20000
#MODE = "decimal"
MODE = "binary"
NUM_DIGITS = 3
COMMUTATIVE_LOSS = False

In [2]:
!git clone https://github.com/annaryzhenkoo/test-time-training-structural-loss
%cd test-time-training-structural-loss

Cloning into 'test-time-training-structural-loss'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 28 (delta 4), reused 25 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 1.02 MiB | 4.17 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/test-time-training-structural-loss


In [2]:
#%cd /content/test-time-training-structural-loss
from src.dataset_generation import dataset_generation

path = dataset_generation(DATASET_SIZE,MODE, NUM_DIGITS)

Dataset size: 20000


In [3]:
from src.data import *

vocab = Vocab(mode = MODE)

In [4]:
len(vocab.symbol2id)

6

In [5]:
vocab.symbol2id

{'0': 0, '1': 1, '=': 2, '+': 3, '<EOS>': 4, '<PAD>': 5}

In [6]:
vocab.PLUS_ID

3

In [7]:
vocab.EQUAL_ID


2

In [8]:
path

'data/data_dn3_ds20000_mbinary.csv'

In [9]:
import pandas as pd

dataset = pd.read_csv(path)
dataset.head()

,0
0,00010011+01111111=011000111
1,1110001111+0000101=11101000001
2,101010001+1111000101=0010010111
3,00111111+1111001111=11010011001
4,1010000111+110010011=00011000101


In [10]:
from sklearn.model_selection import train_test_split

train_dataset, test_dataset = train_test_split(
    dataset,
    test_size=0.2,
    random_state=42
)

In [11]:
len(train_dataset)

16000

In [12]:
len(test_dataset)

4000

In [13]:
from src.model import *

trainDataSet = DatasetSum(train_dataset.iloc[:,0].tolist())

In [14]:
from src.train import *

In [15]:
train_loader = DataLoader(trainDataSet, batch_size=1,
                                collate_fn=partial(collate_fn,vocab=vocab),shuffle=False)

In [16]:
trainDataSet[0]

'111001111+00011001=1111111001'

In [17]:
for i, b in train_loader:
    print(i)
    print(b)
    f

tensor([[0, 1, 1, 1, 3, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 2, 1, 0, 0, 1, 1, 0, 1, 0,
         1, 1, 4]], dtype=torch.int32)
tensor([[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 0, 0, 1, 1, 0, 1, 0,
         1, 1, 4]])


NameError: name 'f' is not defined

In [17]:
vocab.PAD_ID

5

In [ ]:
from src.model import *
from src.train import *

trainDataSet = DatasetSum(train_dataset.iloc[:,0].tolist())
testDataSet = DatasetSum(test_dataset.iloc[:,0].tolist())

if MODE == "binary":
    binary = True
else:
    binary = False

#rnn_addition_simple = RNNAddition(len(vocab.symbol2id), hidden_size=256, binary=binary)
model = CharGRU(vocab_size= len(vocab.symbol2id))
train_model(model=model,trainDataSet=trainDataSet, testDataSet=testDataSet, vocab= vocab, num_epochs=500, exp_name="001",
                patience = 50, trainCommativefunction=COMMUTATIVE_LOSS)

Number of parameters: 63174


  0%|          | 1/500 [00:04<35:26,  4.26s/it]


[1] train_loss=0.9951 train_acc=0.4798 train_em=0.0000
val_loss=0.8496 val_acc=0.4864 val_em=0.0000


 10%|█         | 50/500 [06:24<1:20:54, 10.79s/it]


[50] train_loss=0.5241 train_acc=0.6356 train_em=0.0048
val_loss=0.5249 val_acc=0.6377 val_em=0.0037


 11%|█         | 53/500 [07:01<1:27:45, 11.78s/it]

In [6]:
from src.model import *

model = RNNAddition(len(vocab.symbol2id), hidden_size=256, binary=False)

In [9]:
model.load_state_dict(torch.load(f"outputs/best_rnn.pt",
                                     map_location="cpu"))

<All keys matched successfully>

In [14]:
from src.inference import *

generate_answer(model=model,example="134+201=", vocab=vocab, mode="decimal")

335